In [ ]:
# datasets is installed in the Anaconda base kernel.
# If you ever need to reinstall it from inside the notebook, run:
# %pip install datasets

In [ ]:
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path

print("Loading product dataset...")
try:
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")
    print(f"Loaded {len(dataset)} products")

    products_df = pd.DataFrame(dataset) # type: ignore
    print(f"Dataset columns: {products_df.columns.tolist()}")

except Exception as e:
    print(f"Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")

    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        }
    ]

    products_df = pd.DataFrame(products_data)

images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)

print("\nDataset prepared!")
print(f"  Total products: {len(products_df)}")


In [ ]:

import base64
from io import BytesIO

def encode_pil_image_to_base64(pil_image):
    buffer = BytesIO()
    pil_image.save(buffer, format="JPEG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return encoded

In [ ]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    extra = f"- Additional Info: {additional_info}" if additional_info else ""

    prompt = f"""
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{extra}

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150–200 words)
3. **Key Features** (bullet points, 5–7 items)
4. **SEO Keywords** (comma-separated, 10–15 relevant keywords)

Format your response as JSON:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2"],
    "keywords": "keyword1, keyword2"
}}
"""
    return prompt

In [ ]:
def load_image_from_dataset(row):
    """
    Loads an image from the HuggingFace dataset row.
    Returns a PIL image.
    """
    image_info = row["image"]

    # If dataset provides a URL
    if "url" in image_info and image_info["url"]:
        response = requests.get(image_info["url"])
        return Image.open(BytesIO(response.content)).convert("RGB")

    # If dataset provides a local path
    if "path" in image_info:
        return Image.open(image_info["path"]).convert("RGB")

    raise ValueError("No valid image source found in dataset row.")


In [ ]:
import requests
from PIL import Image
from io import BytesIO

def load_image_from_dataset(row):
    """
    The 'image' field in this dataset is already a PIL Image object.
    """
    img = row["image"]

    if isinstance(img, Image.Image):
        return img.convert("RGB")

    raise ValueError("Row does not contain a PIL image.")

# Pick first product
row = products_df.iloc[0]

# Load image
pil_img = load_image_from_dataset(row)

# Encode image
img_b64 = encode_pil_image_to_base64(pil_img)

# Build prompt
prompt = create_product_listing_prompt(
    product_name=row["productDisplayName"],
    price=float(row.get("price", 0.0)),
    category=row.get("masterCategory", "Unknown"),
    additional_info=row.get("subCategory")  # optional
)

print(prompt[:500], "...")
print("\nEncoded image length:", len(img_b64))


In [ ]:
import json

results = []
errors = []

print("Generating listings for 3 products...\n")

# Process first 3 products
for idx, row in products_df.iloc[:3].iterrows():
    print(f"Processing product {idx+1}/3")

    try:
        # Load image (dataset stores PIL images)
        pil_img = load_image_from_dataset(row)

        # Encode image
        img_b64 = encode_pil_image_to_base64(pil_img)

        # Build prompt
        prompt = create_product_listing_prompt(
            product_name=row.get("productDisplayName", "Unknown Product"),
            price=float(row.get("price", 0.0)),
            category=row.get("masterCategory", "Unknown"),
            additional_info=row.get("subCategory")  # optional
        )

        # Placeholder model response (replace with your API call)
        model_response = {
            "title": "Example title",
            "description": "Example description",
            "features": ["Example feature"],
            "keywords": "example, keywords"
        }

        # Save result
        results.append({
            "id": row.get("id", idx),
            "prompt": prompt,
            "image_base64": img_b64,
            "listing": model_response
        })

        print("✓ Success\n")

    except Exception as e:
        print(f"✗ Error: {e}\n")
        errors.append({"index": idx, "error": str(e)})
        continue


In [ ]:
import json

results = []
errors = []

print(f"Starting batch processing for {len(products_df)} products...\n")

for idx, row in products_df.iterrows():
    print(f"Processing product {idx+1}/{len(products_df)}")

    try:
        # Load image (dataset already stores PIL images)
        pil_img = load_image_from_dataset(row)

        # Encode image
        img_b64 = encode_pil_image_to_base64(pil_img)

        # Build prompt
        prompt = create_product_listing_prompt(
            product_name=row.get("productDisplayName", "Unknown Product"),
            price=float(row.get("price", 0.0)),
            category=row.get("masterCategory", "Unknown"),
            additional_info=row.get("subCategory")  # optional
        )

        # Placeholder model response (replace with your API call)
        model_response = {
            "title": "Example title",
            "description": "Example description",
            "features": ["Example feature"],
            "keywords": "example, keywords"
        }

        # Save result
        results.append({
            "id": row.get("id", idx),
            "prompt": prompt,
            "image_base64": img_b64,
            "listing": model_response
        })

        print("✓ Success\n")

    except Exception as e:
        print(f"✗ Error: {e}\n")
        errors.append({"index": idx, "error": str(e)})
        continue


In [ ]:
with open("product_listings.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)
